In [ ]:
from rdflib import Graph, Namespace, RDF, Literal

# 1. Laden der beiden Graphen
source_graph = Graph()
target_graph = Graph()

print("Lade Dateien...")
# Passen Sie die Pfade ggf. an
source_graph.parse(r"D:\MA_Python_Agent\MSRGuard_Anpassung\KGs\TestSIM_filled.ttl", format="turtle")
target_path = r"D:\MA_Python_Agent\MSRGuard_Anpassung\KGs\FMEA_KG_FischertechnikI4.0-Simulator.ttl"
target_graph.parse(target_path, format="turtle")

# 2. Definieren der Namespaces basierend auf Ihren Dateien
# Namespace für die PLC-Parameter (aus Test2_filled.ttl)
DP_PLC = Namespace("http://www.semanticweb.org/AgentProgramParams/dp_")

# Namespace für die FMEA-Klassen (aus FMEA_KG.ttl)
# Hinweis: In Ihrer Datei ist der Prefix cl: definiert als .../class_
CL_FMEA = Namespace("http://www.semanticweb.org/FMEA_VDA_AIAG_2021/class_")

# Zähler für die Statistik
sensor_count = 0
actor_count = 0

print("Starte Übertragung...")

# 3. Iterieren über alle Subjekte, die eine Hardware-Adresse haben
# Wir suchen nach Tripeln (Subjekt, Prädikat, Objekt), wobei das Prädikat hasHardwareAddress ist
for subject, predicate, obj in source_graph.triples((None, DP_PLC.hasHardwareAddress, None)):
    
    # Das Objekt ist hier der String der Adresse (z.B. "IX0.0" oder "QX0.0")
    address_str = str(obj)
    
    # 4. Logik für Aktoren (Q) und Sensoren (I)
    if "Q" in address_str:
        # Füge das Subjekt als Instanz der Klasse 'Actors' in den Zielgraphen ein
        target_graph.add((subject, RDF.type, CL_FMEA.Actors))
        actor_count += 1
        print(f"Aktor hinzugefügt: {subject} (Adresse: {address_str})")
        
    elif "I" in address_str:
        # Füge das Subjekt als Instanz der Klasse 'Sensors' in den Zielgraphen ein
        target_graph.add((subject, RDF.type, CL_FMEA.Sensors))
        sensor_count += 1
        print(f"Sensor hinzugefügt: {subject} (Adresse: {address_str})")

# 5. Speichern des aktualisierten FMEA-Graphen
target_graph.serialize(destination=target_path, format="turtle")

print("-" * 30)
print(f"Fertig! Übertragen: {sensor_count} Sensoren und {actor_count} Aktoren.")
print(f"Ergebnis gespeichert in: {target_path}")

Lade Dateien...
Starte Übertragung...
Aktor hinzugefügt: http://www.semanticweb.org/AgentProgramParams/GVL_HRL__dot__HRL_K1_1_Zaehler_Zuruecksetzen (Adresse: %Q*)
Aktor hinzugefügt: http://www.semanticweb.org/AgentProgramParams/GVL_HRL__dot__HRL_K1_2_Zaehler_Zuruecksetzen (Adresse: %Q*)
Aktor hinzugefügt: http://www.semanticweb.org/AgentProgramParams/GVL_HRL__dot__HRL_MOT_Ausleger_rueckwaerts (Adresse: %Q*)
Aktor hinzugefügt: http://www.semanticweb.org/AgentProgramParams/GVL_HRL__dot__HRL_MOT_Ausleger_vorwaerts (Adresse: %Q*)
Aktor hinzugefügt: http://www.semanticweb.org/AgentProgramParams/GVL_HRL__dot__HRL_MOT_Foerderband_rueckwaerts (Adresse: %Q*)
Aktor hinzugefügt: http://www.semanticweb.org/AgentProgramParams/GVL_HRL__dot__HRL_MOT_Foerderband_vorwaerts (Adresse: %Q*)
Aktor hinzugefügt: http://www.semanticweb.org/AgentProgramParams/GVL_HRL__dot__HRL_MOT_horizontal_zum_Foerderband (Adresse: %Q*)
Aktor hinzugefügt: http://www.semanticweb.org/AgentProgramParams/GVL_HRL__dot__HRL_MOT_ho

In [2]:
from pathlib import Path
import sys

ROOT = Path(r"D:\MA_Python_Agent")
NOTEBOOK_DIR = ROOT / "Notebooks"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from fmea_skill_function_builder import FMEASkillFunctionBuilder

skill_function_builder = FMEASkillFunctionBuilder(
    plc_kg_path=ROOT / "MSRGuard_Anpassung" / "KGs" / "TestSIM_filled.ttl",
    fmea_kg_path=ROOT / "MSRGuard_Anpassung" / "KGs" / "FMEA_KG_FischertechnikI4.0-Simulator.ttl",
    asrs_kg_path=Path(r"C:\Users\Alexander Verkhov\Downloads\ASRS_KB1_v0.62.owl"),
)

skill_function_suggestions = skill_function_builder.suggest_functions()
skill_function_builder.preview(skill_function_suggestions, limit=50)

# Write explicitly after checking the preview:
# skill_function_builder.add_functions(skill_function_suggestions, output_path=skill_function_builder.fmea_kg_path)

# Vorschläge als SkillImplementationHypothesis in TestSIM_filled.ttl schreiben

created, updated = skill_function_builder.write_hypotheses_to_plc_kg(skill_function_suggestions)
print(f"Hypothesen gespeichert in: {skill_function_builder.plc_kg_path}")
print(f"  erstellt={created}, aktualisiert={updated}")


Skill-Function-Vorschlaege: 42 angezeigt
- BSkill_MoveLoadCarrier_Bwd | parent=CSkill_HRL_ASRS | runtime=HRL_CB_AS_HorizontalMoveSensors, HRL_Skill_CB_MoveBackwards | conf=0.85 | SkillSet fallback + ASRS map
- BSkill_MoveLoadCarrier_Fwd | parent=CSkill_HRL_ASRS | runtime=HRL_CB_AS_HorizontalMoveSensors, HRL_Skill_CB_MoveForwards | conf=0.85 | SkillSet fallback + ASRS map
- BSkill_MoveSRM_X_Bwd | parent=CSkill_HRL_ASRS | runtime=HRL_RGB_AS_HorizontalMoveEncoders, HRL_Skill_RGB_MoveHorizontalBackwards | conf=0.85 | SkillSet fallback + ASRS map
- BSkill_MoveSRM_X_Fwd | parent=CSkill_HRL_ASRS | runtime=HRL_RGB_AS_HorizontalMoveEncoders, HRL_Skill_RGB_MoveHorizontalForwards | conf=0.85 | SkillSet fallback + ASRS map
- BSkill_MoveSRM_Y_Bwd | parent=CSkill_HRL_ASRS | runtime=HRL_RGB_AS_VerticalMoveEncoders, HRL_Skill_RGB_MoveVerticalBackwards | conf=0.85 | SkillSet fallback + ASRS map
- BSkill_MoveSRM_Y_Fwd | parent=CSkill_HRL_ASRS | runtime=HRL_RGB_AS_VerticalMoveEncoders, HRL_Skill_RGB_Move

In [ ]:
# Alle Suggestions als class_Skill + op:implementsSkill + op:isSubSkillOf
# + op:validatedFromHypothesis in die FMEA-KG schreiben.

from rdflib import Graph, Namespace, Literal, RDF, RDFS, OWL

AGENT_BASE = "http://www.semanticweb.org/AgentProgramParams/"
AG = Namespace(AGENT_BASE)
DP = Namespace(f"{AGENT_BASE}dp_")
OP = Namespace(f"{AGENT_BASE}op_")

plc_path = skill_function_builder.plc_kg_path
g = Graph()
g.parse(str(plc_path), format="turtle")

# Schema fuer neue Object Properties
for prop, domain, range_ in [
    (OP.isSubSkillOf,          AG.class_Skill, AG.class_Skill),
    (OP.validatedFromHypothesis, AG.class_Skill, AG.class_SkillImplementationHypothesis),
]:
    g.add((prop, RDF.type, OWL.ObjectProperty))
    g.add((prop, RDFS.domain, domain))
    g.add((prop, RDFS.range, range_))

created_skills = created_links = 0

for s in skill_function_suggestions:
    skill_uri = AG[s.function_id]
    hyp_uri   = AG[f"Hyp_{s.function_id}"]

    # class_Skill Individual
    if (skill_uri, RDF.type, AG.class_Skill) not in g:
        g.add((skill_uri, RDF.type, OWL.NamedIndividual))
        g.add((skill_uri, RDF.type, AG.class_Skill))
        g.add((skill_uri, RDFS.label, Literal(s.label)))
        if s.description:
            g.add((skill_uri, DP.hasSkillDescription, Literal(s.description)))
        created_skills += 1

    # Skill -> Hypothese (Rueckverfolgbarkeit)
    g.add((skill_uri, OP.validatedFromHypothesis, hyp_uri))

    # POU op:implementsSkill Skill
    for pou_iri in s.pou_iris:
        if (pou_iri, OP.implementsSkill, skill_uri) not in g:
            g.add((pou_iri, OP.implementsSkill, skill_uri))
            created_links += 1

    # Skill op:isSubSkillOf ParentSkill
    if s.parent_function_id:
        parent_uri = AG[s.parent_function_id]
        if (parent_uri, RDF.type, AG.class_Skill) not in g:
            g.add((parent_uri, RDF.type, OWL.NamedIndividual))
            g.add((parent_uri, RDF.type, AG.class_Skill))
            g.add((parent_uri, RDFS.label, Literal(s.parent_function_id.replace("_", " "))))
        g.add((skill_uri, OP.isSubSkillOf, parent_uri))

g.serialize(destination=str(plc_path), format="turtle")
print(f"Gespeichert in: {plc_path}")
print(f"  neue Skills: {created_skills}, implementsSkill-Tripel: {created_links}")


In [ ]:
# Zelle 4: Transitive Analyse Skill -> Sensoren/Aktoren im PLC-KG
from rdflib import Graph
from fmea_skill_function_builder import SkillHardwareIOResolver

plc_path = skill_function_builder.plc_kg_path
g_plc = Graph()
g_plc.parse(str(plc_path), format="turtle")

io_resolver = SkillHardwareIOResolver(g_plc)
skill_io_results = io_resolver.resolve_all(skill_function_suggestions)
skill_io_map = {
    function_id: result.as_skill_io_entry()
    for function_id, result in skill_io_results.items()
}

for s in skill_function_suggestions:
    result = skill_io_results[s.function_id]
    tag = f"{len(result.sensors)}S / {len(result.actors)}A"
    status = tag if result.has_hardware else "(keine HW-Zuordnung)"
    print(
        f"{s.function_id:<45} {status}  "
        f"seeds={len(result.seed_calls)} visited={result.visited_nodes}"
    )


In [ ]:
# Zelle 5: Skills als ProcessFunctions in FMEA einfuegen
# Verbindung: Sensor/Aktor op:hasFunction ProcessFunction
# RuntimeSkillName wird direkt aus dem SPS-KG abgeleitet:
#   FBType/POU --op:implementsSkill--> Skill
#   FBType/POU --dp:hasPOUName-------> RuntimeSkillName
from rdflib import Graph, Namespace, Literal, RDF, RDFS, OWL, XSD

AGENT_BASE = "http://www.semanticweb.org/AgentProgramParams/"
FMEA_BASE = "http://www.semanticweb.org/FMEA_VDA_AIAG_2021/"

AG      = Namespace(AGENT_BASE)
DP_PLC  = Namespace(f"{AGENT_BASE}dp_")
OP_PLC  = Namespace(f"{AGENT_BASE}op_")
CL      = Namespace(f"{FMEA_BASE}class_")
OP_FMEA = Namespace(f"{FMEA_BASE}op_")
DP_FMEA = Namespace(f"{FMEA_BASE}dp_")
FMEA    = Namespace(FMEA_BASE)

def local_name(uri):
    text = str(uri)
    if "#" in text:
        return text.rsplit("#", 1)[-1]
    return text.rstrip("/").rsplit("/", 1)[-1]

plc_path = skill_function_builder.plc_kg_path
g_plc = Graph()
g_plc.parse(str(plc_path), format="turtle")

fmea_path = skill_function_builder.fmea_kg_path
g_fmea = Graph()
g_fmea.parse(str(fmea_path), format="turtle")

# Schema absichern, falls die Property im Ziel-KG noch nicht deklariert ist.
g_fmea.add((DP_FMEA.hasRuntimeSkillName, RDF.type, OWL.DatatypeProperty))
g_fmea.add((DP_FMEA.hasRuntimeSkillName, RDFS.domain, CL.Function))
g_fmea.add((DP_FMEA.hasRuntimeSkillName, RDFS.range, XSD.string))

# Mapping aus SPS-KG: Skill-Individual -> RuntimeSkillName (= POU/FBType-Name).
runtime_names_by_skill_id = {}
for fbtype, _, skill in g_plc.triples((None, OP_PLC.implementsSkill, None)):
    skill_id = local_name(skill)
    runtime_name = next(g_plc.objects(fbtype, DP_PLC.hasPOUName), None)
    runtime_name = str(runtime_name or local_name(fbtype)).strip()
    if runtime_name:
        runtime_names_by_skill_id.setdefault(skill_id, set()).add(runtime_name)

skill_io_map_for_cell = globals().get("skill_io_map", {})
created_funcs = created_links = skipped_hw = runtime_links = missing_runtime = 0
missing_runtime_examples = []

for s in skill_function_suggestions:
    io = skill_io_map_for_cell.get(s.function_id, {})
    sensors = io.get("sensors", set())
    actors  = io.get("actors",  set())
    elements = sensors | actors

    # ProcessFunction Individual (in FMEA-Namespace)
    func_uri = FMEA[s.function_id]
    if (func_uri, RDF.type, CL.ProcessFunction) not in g_fmea:
        g_fmea.add((func_uri, RDF.type, OWL.NamedIndividual))
        g_fmea.add((func_uri, RDF.type, CL.Function))
        g_fmea.add((func_uri, RDF.type, CL.ProcessFunction))
        g_fmea.add((func_uri, RDFS.label, Literal(s.label)))
        if s.description:
            g_fmea.add((func_uri, DP_FMEA.hasFunctionDescription, Literal(s.description)))
        created_funcs += 1

    # RuntimeSkillName: ausschliesslich aus SPS-KG implementsSkill + hasPOUName.
    g_fmea.remove((func_uri, DP_FMEA.hasRuntimeSkillName, None))
    runtime_names = sorted(runtime_names_by_skill_id.get(s.function_id, set()))
    if runtime_names:
        for runtime_name in runtime_names:
            g_fmea.add((func_uri, DP_FMEA.hasRuntimeSkillName, Literal(runtime_name, datatype=XSD.string)))
            runtime_links += 1
    else:
        missing_runtime += 1
        if len(missing_runtime_examples) < 10:
            missing_runtime_examples.append(s.function_id)

    if not elements:
        skipped_hw += 1
        continue  # Keine Sensor-/Aktor-Verbindung, Function bleibt trotzdem im FMEA-KG.

    # Element op:hasFunction ProcessFunction
    for elem in elements:
        if (elem, OP_FMEA.hasFunction, func_uri) not in g_fmea:
            g_fmea.add((elem, OP_FMEA.hasFunction, func_uri))
            created_links += 1

g_fmea.serialize(destination=str(fmea_path), format="turtle")
print(f"Gespeichert in: {fmea_path}")
print(f"  neue ProcessFunctions : {created_funcs}")
print(f"  dp:hasRuntimeSkillName: {runtime_links}")
print(f"  op:hasFunction-Tripel : {created_links}")
print(f"  ohne HW-Bezug         : {skipped_hw}")
print(f"  ohne Runtime-Mapping  : {missing_runtime}")
if missing_runtime_examples:
    print("  Beispiele ohne Runtime-Mapping:", ", ".join(missing_runtime_examples))
